# 1.0 — AutoFT baseline on iWildCam (WILDS leaderboard #1 reproduction)

Native reimplementation of **AutoFT: Learning an Objective for Robust Fine-tuning** (Choi et al.,
arXiv 2401.10220; official code: https://github.com/cchoi1/auto-ft) in `iwild/modeling/`,
following the Cookiecutter Data Science conventions (Typer CLIs under the hood, paths from
`iwild.config`).

**Reference numbers (OOD test macro F1, official WILDS leaderboard):**

| Entry | Backbone | Macro F1 |
|---|---|---|
| AutoFT (ours to reproduce) | CLIP ViT-L/14@336 + WiSE-FT | 52.0 (0.4) |
| FLYP | CLIP ViT-L/14@336 | 46.0 (1.3) |
| Model Soups | CLIP ViT-L/14@336 | 43.3 |
| ERM | CLIP ViT-L/14@336 | 41.4 |

This workstation has a single RTX 4070 (8 GB), so the reproduction target is **CLIP ViT-B/16 @
224px** (the leaderboard number above is ViT-L/14@336 — sanctioned by the project plan doc;
our absolute numbers will sit lower and are reported *relative to the FLYP baseline on the same
backbone*, never as leaderboard entries).

**Documented deviations from the official repo** (all else — loss formulas, search-space
priors, objective, early-stopping metric, prompts, CSV duplication — matches the official
recipe): pandas instead of dask for the FT CSV; `state_dict` checkpoints instead of
whole-module pickles; no DataParallel (single GPU); batch 64×accum-2 vs 128×2 (VRAM);
`--hopt-evals` default 100 vs the paper's 500; class-balanced OOD-HP subset actually applied
(the official `--use_class_balanced_ood` flag is parsed but never wired).

Protocol: the classifier trains on the **full official train split** (leaderboard-faithful);
hyperparameters are tuned on a 1000-example class-balanced subset of the official OOD val;
OOD test is evaluated once, at the end.

## 0. Environment check

Run this notebook from the project `.venv` (the one `make requirements-train` populated).

In [ ]:
import sys
from pathlib import Path

import torch

assert Path(sys.executable).parent.name == "Scripts", "run Jupyter from the project .venv"
print("python:", sys.version)
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 1. Download + verify the dataset

~12 GB compressed, fetched directly from the CodaLab bundle (certificate-less
endpoint first, no-cert-verification https fallback — the bundle's TLS cert is
currently broken). Verification asserts the split sizes, class count, camera-trap
counts, and the **leakage rule** (no trap crosses train/val/test slabs).

Manual fallback: https://worksheets.codalab.org/bundles/0x6313da2b204647e79a14b468131fcd64

In [ ]:
!{sys.executable} iwild/dataset.py download

## 2. Build the FLYP fine-tuning CSV + the location split

The FT CSV has one row per (train image × 2 text prompts) = 259,618 rows, tab-separated
`title,filepath,label` — this is exactly the official auto-ft `datacreation_scripts/iwildcam.py`
recipe. The location split (80/20 of the 243 train traps) is for the later CF-Δ selector lane;
the AutoFT baseline itself trains on the full official train split.

In [ ]:
!{sys.executable} iwild/dataset.py make-ft-csv
!{sys.executable} iwild/dataset.py location-split --seed 42

## 3. Zero-shot CLIP model (ViT-B/16)

Builds the 182-way zero-shot head from the two iWildCam prompts per class (rows of
`labels.csv` with y < 99999, matching the official auto-ft code and WILDS v2.0's 182 classes).

In [ ]:
!{sys.executable} iwild/modeling/train.py zeroshot

## 4. Sanity: zero-shot macro F1 on the OOD val split

A healthy ViT-B/16 zero-shot model should land in the tens-of-percent macro F1 range. Anything
near zero (or NaN) means something is wrong with the head/prompts before we spend GPU time.

In [ ]:
import json

from iwild.config import REPORTS_DIR, ZEROSHOT_DIR

ckpt = ZEROSHOT_DIR / "clip_vitb16_iwildcam.pt"
!{sys.executable} iwild/modeling/train.py eval --ckpt {ckpt}
report = json.loads((REPORTS_DIR / f"autoft_eval_{ckpt.stem}.json").read_text())
print(json.dumps({k: round(v["F1-macro_all"], 3) for k, v in report.items()}, indent=2))

## 5. Smoke AutoFT run (minutes)

End-to-end check of the outer loop + final fine-tune at toy scale (3 trials × 5 inner steps,
batch 16, 200-example OOD-HP subset, 2000 FT rows, 1 epoch). Must complete without OOM before
the real run is launched.

In [ ]:
!{sys.executable} iwild/modeling/train.py autoft --smoke --run-name smoke --no-init-hparams

## 6. Full AutoFT run (staged + resumable)

Run the two stages as separate commands so a crash or reboot only costs the current
stage; everything lands under `models/autoft/<run-name>/`.

**Stage A — hyperparameter search** (`--stage hopt`). Official recipe: 500 Optuna trials x
100 inner steps (paper README); on the RTX 4070 downscale to `--hopt-evals 100
--inner-steps 10`. Trial 0 is seeded with the pre-learned weights shipped at
`iwild/modeling/autoft_hparams_iwildcam.json` (learned by the authors on ViT-B/16, then
transferred to ViT-L/14@336 for the leaderboard entry). The objective is macro F1 on a
class-balanced 1000-example subset of the official OOD val. Writes `hparams.json`.

**Stage B — final fine-tune** (`--stage ft --resume`): 20 epochs over the 259,618-row FLYP
CSV, best epoch kept by ID-val macro F1. Per-epoch checkpoints (`epoch_ckpt/last.pt`,
`best.pt`, `progress.json`) make it resumable — re-run the same command to continue after
an interruption. Fast path: skip Stage A by copying the paper's hparams JSON into the run
directory as `hparams.json`.

```
python iwild/modeling/train.py autoft --stage hopt --hopt-evals 100 --inner-steps 10 --run-name autoft_vitb16
python iwild/modeling/train.py autoft --stage ft --ft-epochs 20 --resume --run-name autoft_vitb16
python iwild/modeling/train.py flyp --run-name flyp_vitb16 --resume   # lower baseline, same backbone
```

After Stage B, WiSE-FT weight interpolation (the ensembling step behind the 52.0 leaderboard
number) is a cheap post-hoc:

```
python iwild/modeling/train.py wisefmt --finetuned models/autoft/autoft_vitb16/final_model.pt
```

Add `--wandb` (+ optional `--wandb-project` / `--wandb-entity`, else the `WANDB_PROJECT` /
`WANDB_ENTITY` env vars) to log trials, epochs, and evals to Weights & Biases.

## 7. Final evaluation (OOD test touched once)

Report macro F1 / accuracy on all four held-out splits. Compare against the FLYP run on the
same backbone — the project claims are *relative to the reproduced baselines under identical
conditions*.

In [ ]:
final_ckpt = Path("models/autoft/autoft_vitb16/final_model.pt")  # noqa: after Stage B
assert final_ckpt.exists(), "run Stage A/B first"
!{sys.executable} iwild/modeling/train.py eval --ckpt {final_ckpt}
report = json.loads((REPORTS_DIR / f"autoft_eval_{final_ckpt.stem}.json").read_text())
for split, metrics in report.items():
    print(f"{split:8s}  macro F1 = {metrics['F1-macro_all']:.3f}  acc = {metrics['acc']:.2f}")